# Quantized CNN Training for Posture Classification (HGQ2-based QAT)

**Dataset**
- **Input:** 8×8 pressure maps (64 features per sample)
- **Output:** 6 classes (fetus_L, fetus_R, prone, supine, trunk_L, trunk_R)
- **Raw data:** Float32 values in range [0, 235.9]

**Architecture**
- 3 QConv2D layers with QBatchNormalization
- Filter progression: [16, 16, 24]
- GlobalAveragePooling2D for efficiency
- Output: QDense(6) + QSoftmax

**Quantization Strategy (HGQ2)**
- Automatic bitwidth optimization via gradient descent
- `kbi` quantizer for weights (SAT_SYM overflow)
- `kif` quantizer for activations (WRAP overflow)
- Effective Bit-Operations (EBOPs) tracking for resource estimation

**Hyperparameters**
- Learning rate: 0.001
- Batch size: 32
- Kernel size: (3, 3)
- L1 regularization: 1e-4
- Early stopping patience: 15
- Max epochs: 50

## Imports

In [13]:
import os
import numpy as np
import matplotlib.pyplot as plt
import time
from pathlib import Path

# Keras 3
import keras
from keras import layers
from keras import regularizers
from keras import optimizers
from keras.utils import to_categorical

# HGQ2 quantization-aware training
from hgq.layers import QConv2D, QDense, QBatchNormalization
from hgq.config import LayerConfigScope, QuantizerConfigScope
from hgq.utils.sugar import FreeEBOPs
from hgq.utils import trace_minmax

# hls4ml for hardware conversion
import hls4ml

# Scikit-learn for metrics
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

# Set random seeds for reproducibility
np.random.seed(42)
keras.utils.set_random_seed(42)

## Data Loading

In [14]:
# Dataset paths
dataset_dir = Path('../dataset/output')
train_file = dataset_dir / 'train_raw.npz'
val_file = dataset_dir / 'val_raw.npz'
test_file = dataset_dir / 'test_raw.npz'

# Load datasets
train_data = np.load(train_file, allow_pickle=True)
val_data = np.load(val_file, allow_pickle=True)
test_data = np.load(test_file, allow_pickle=True)

# Extract features and labels
X_train_raw = train_data['data']
y_train_raw = train_data['labels']

X_val_raw = val_data['data']
y_val_raw = val_data['labels']

X_test_raw = test_data['data']
y_test_raw = test_data['labels']

print("\nDataset shapes:")
print(f"  Train: {X_train_raw.shape}, labels: {y_train_raw.shape}")
print(f"  Val:   {X_val_raw.shape}, labels: {y_val_raw.shape}")
print(f"  Test:  {X_test_raw.shape}, labels: {y_test_raw.shape}")


Dataset shapes:
  Train: (40325, 64), labels: (40325,)
  Val:   (20162, 64), labels: (20162,)
  Test:  (40326, 64), labels: (40326,)


## Data Preprocessing

In [15]:
# Encode labels as integers 0-5
label_encoder = LabelEncoder()
y_train_int = label_encoder.fit_transform(y_train_raw)
y_val_int = label_encoder.transform(y_val_raw)
y_test_int = label_encoder.transform(y_test_raw)

print("Label encoding:")
for i, label in enumerate(label_encoder.classes_):
    print(f"  {label} -> {i}")

# One-hot encode labels
num_classes = 6
y_train = to_categorical(y_train_int, num_classes=num_classes)
y_val = to_categorical(y_val_int, num_classes=num_classes)
y_test = to_categorical(y_test_int, num_classes=num_classes)

print(f"\nOne-hot encoded shape: {y_train.shape[1]}")

Label encoding:
  fetus_L -> 0
  fetus_R -> 1
  prone -> 2
  supine -> 3
  trunk_L -> 4
  trunk_R -> 5

One-hot encoded shape: 6


In [16]:
# Reshape data from (N, 64) to (N, 8, 8, 1)
X_train = X_train_raw.reshape(-1, 8, 8, 1).astype('float32')
X_val = X_val_raw.reshape(-1, 8, 8, 1).astype('float32')
X_test = X_test_raw.reshape(-1, 8, 8, 1).astype('float32')

print("Reshaped data:")
print(f"  Train: {X_train.shape}")
print(f"  Val:   {X_val.shape}")
print(f"  Test:  {X_test.shape}")
print(f"\nData range: [{X_train.min():.4f}, {X_train.max():.4f}]")
print(f"Data dtype: {X_train.dtype}")
print("\nNote: No normalization applied - HGQ2 handles input range automatically via input quantizer.")

Reshaped data:
  Train: (40325, 8, 8, 1)
  Val:   (20162, 8, 8, 1)
  Test:  (40326, 8, 8, 1)

Data range: [0.0000, 235.8969]
Data dtype: float32

Note: No normalization applied - HGQ2 handles input range automatically via input quantizer.


## Model Definition

### HGQ2 Quantization Configuration

HGQ2 uses **gradient-based automatic bitwidth optimization**. Key configuration:

- **`kbi` quantizer** for weights: Optimizes bit-width + integer bits with symmetric saturation overflow
- **`kif` quantizer** for activations (datalane): Optimizes integer bits for data range with wrap-around overflow
- **`beta0=1e-5`**: Controls EBOP regularization strength (lower = more aggressive quantization)
- **`enable_ebops=True`**: Track Effective Bit-Operations for resource estimation

In [17]:
def create_quantized_model(input_shape=(8, 8, 1), num_classes=6):
    """
    Create quantized CNN model for posture classification with HGQ2 layers.

    Architecture matches the base FP32 model but uses HGQ2 quantized layers:
    - QConv2D with QBatchNormalization instead of Conv2D + BatchNormalization
    - QDense for output layer
    - ReLU activations (quantization handled by HGQ2)

    The model is built within a QuantizerConfigScope context that sets:
    - kbi quantizer with SAT_SYM for weights
    - kif quantizer with WRAP for activations
    - EBOP tracking enabled

    HGQ2 automatically optimizes bitwidths during training via gradients.
    No manual bit-width specification required.

    Args:
        input_shape: Input tensor shape (H, W, C), default (8, 8, 1)
        num_classes: Number of output classes, default 6

    Returns:
        Keras Model instance with quantized layers
    """
    # Input layer
    inputs = keras.Input(shape=input_shape, name='pressure_map')

    # Conv Block 1: 16 filters, 3x3 kernel
    x = QConv2D(
        filters=16,
        kernel_size=(3, 3),
        kernel_initializer='lecun_uniform',
        kernel_regularizer=regularizers.l1(1e-4),
        use_bias=False,
        name='qconv_1'
    )(inputs)
    # x = QBatchNormalization(name='qbn_1')(x)
    x = keras.layers.Activation('relu', name='conv_act_1')(x)

    # Conv Block 2: 16 filters, 3x3 kernel
    x = QConv2D(
        filters=16,
        kernel_size=(3, 3),
        kernel_initializer='lecun_uniform',
        kernel_regularizer=regularizers.l1(1e-4),
        use_bias=False,
        name='qconv_2'
    )(x)
    # x = QBatchNormalization(name='qbn_2')(x)
    x = keras.layers.Activation('relu', name='conv_act_2')(x)

    # Conv Block 3: 24 filters, 3x3 kernel
    x = QConv2D(
        filters=24,
        kernel_size=(3, 3),
        kernel_initializer='lecun_uniform',
        kernel_regularizer=regularizers.l1(1e-4),
        use_bias=False,
        name='qconv_3'
    )(x)
    # x = QBatchNormalization(name='qbn_3')(x)
    x = keras.layers.Activation('relu', name='conv_act_3')(x)

    # Pooling
    x = keras.layers.MaxPooling2D(pool_size=(2, 2), name='pool_3')(x)
    x = keras.layers.GlobalAveragePooling2D(name='global_avg_pool')(x)

    # Output layer: 6 classes with softmax
    # QDense includes built-in output quantizer for activations
    outputs = QDense(num_classes, activation='softmax', name='output')(x)

    # Create model
    model = keras.Model(inputs=inputs, outputs=outputs, name='posture_classifier_qat')

    return model

In [18]:
# Build model with quantization configuration
# Using nested QuantizerConfigScope for different places

print("Creating quantized model with HGQ2 configuration...")
print("\nQuantizer configuration:")
print("  - Weights: kbi quantizer, SAT_SYM overflow, RND rounding")
print("  - Activations (datalane): kif quantizer, WRAP overflow, RND rounding")
print("  - EBOP tracking enabled with beta0=1e-5")

with (
    # Weight quantizer: kbi type with symmetric saturation
    QuantizerConfigScope(
        q_type='kbi',
        place='weight',
        overflow_mode='SAT_SYM',
        round_mode='RND'
    ),
    # Activation quantizer: kif type with wrap-around (tracing required later)
    QuantizerConfigScope(
        q_type='kif',
        place='datalane',
        overflow_mode='WRAP',
        round_mode='RND'
    ),
    # Layer configuration: enable EBOP tracking
    LayerConfigScope(
        enable_ebops=True,
        beta0=1e-5
    ),
):
    model = create_quantized_model()

print("\nModel created successfully.")
model.summary()

Creating quantized model with HGQ2 configuration...

Quantizer configuration:
  - Weights: kbi quantizer, SAT_SYM overflow, RND rounding
  - Activations (datalane): kif quantizer, WRAP overflow, RND rounding
  - EBOP tracking enabled with beta0=1e-5

Model created successfully.


Model: "posture_classifier_qat"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ pressure_map (InputLayer)       │ (None, 8, 8, 1)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ qconv_1 (QConv2D)               │ (None, 6, 6, 16)       │           771 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_act_1 (Activation)         │ (None, 6, 6, 16)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ qconv_2 (QConv2D)               │ (None, 4, 4, 16)       │        10,947 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_act_2 (Activation)         │ (None, 4, 4, 16)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ qconv_3 (QConv2D)               │ (None, 2, 2, 24)       │        14,595 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_act_3 (Activation)         │ (None, 2, 2, 24)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool_3 (MaxPooling2D)           │ (None, 1, 1, 24)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_avg_pool                 │ (None, 24)             │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (QDense)                 │ (None, 6)              │           676 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 26,989 (84.99 KB)

 Trainable params: 19,076 (74.52 KB)

 Non-trainable params: 7,913 (10.48 KB)

In [19]:
# Check trainable parameters per layer (for Vivado HLS constraint)
def check_layer_trainable_params(model):
    """Check if any layer exceeds Vivado's 4096 parameter limit for parallel compilation."""
    print("\nLayer parameter counts:")
    total_params = 0
    for layer in model.layers:
        if hasattr(layer, 'get_weights') and len(layer.get_weights()) > 0:
            params = sum(np.prod(w.shape) for w in layer.get_weights())
            if params > 0:
                total_params += params
                status = " ⚠️ TOO LARGE" if params > 4096 else ""
                print(f"  {layer.name}: {params:,} params{status}")
    print(f"\nTotal trainable parameters: {total_params:,}")
    if total_params > 4096:
        print("\nNote: Some layers exceed 4096 parameters.")
        print("This is fine for training, but may require 'reuse_factor > 1' in hls4ml.")

check_layer_trainable_params(model)


Layer parameter counts:
  qconv_1: 771.0 params
  qconv_2: 10,947.0 params ⚠️ TOO LARGE
  qconv_3: 14,595.0 params ⚠️ TOO LARGE
  output: 676.0 params

Total trainable parameters: 26,989.0

Note: Some layers exceed 4096 parameters.
This is fine for training, but may require 'reuse_factor > 1' in hls4ml.


## Training Configuration

In [20]:
# Compile model
model.compile(
    optimizer=optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Model compiled successfully.")

Model compiled successfully.


In [21]:
# Define callbacks
# FreeEBOPs: HGQ2 callback to monitor Effective Bit-Operations during training
ebops_callback = FreeEBOPs()

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=15,  # Higher patience for QAT convergence
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,  # Higher patience for QAT
        verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        filepath='best_quantized_model.keras',
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    ),
    ebops_callback  # Track EBOPs for resource estimation
]

print("Callbacks configured:")
for cb in callbacks:
    print(f"  - {cb.__class__.__name__}")

Callbacks configured:
  - EarlyStopping
  - ReduceLROnPlateau
  - ModelCheckpoint
  - FreeEBOPs


In [22]:
# Training function
def train_model(model, X_train, y_train, X_val, y_val, batch_size=32, max_epochs=50):
    """
    Train the quantized model with QAT.

    QAT typically requires more epochs than FP32 training to converge
    because the quantization parameters (bitwidths) are optimized jointly
    with the weights.

    Args:
        model: Keras model to train
        X_train, y_train: Training data
        X_val, y_val: Validation data
        batch_size: Batch size for training
        max_epochs: Maximum number of epochs

    Returns:
        Training history
    """
    print(f"\nStarting QAT training for up to {max_epochs} epochs...")
    print(f"Batch size: {batch_size}")
    print(f"Training samples: {X_train.shape[0]}")
    print(f"Validation samples: {X_val.shape[0]}")
    print("\nNote: EBOPs (Effective Bit-Operations) estimate FPGA resource usage.")
    print("Lower EBOPs = fewer hardware resources required.\n")

    start_time = time.time()

    history = model.fit(
        X_train, y_train,
        batch_size=batch_size,
        epochs=max_epochs,
        validation_data=(X_val, y_val),
        callbacks=callbacks,
        verbose=1
    )

    training_time = time.time() - start_time
    print(f"\nTraining completed in {training_time/60:.2f} minutes")

    return history

In [23]:
# Plot training history
def plot_training_history(history, save_path=None):
    """Plot training and validation loss/accuracy curves."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Plot loss
    axes[0].plot(history.history['loss'], label='Training Loss', linewidth=2)
    axes[0].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
    axes[0].set_xlabel('Epoch', fontsize=12)
    axes[0].set_ylabel('Loss', fontsize=12)
    axes[0].set_title('Training and Validation Loss (QAT)', fontsize=14)
    axes[0].legend(fontsize=11)
    axes[0].grid(True, alpha=0.3)

    # Plot accuracy
    axes[1].plot(history.history['accuracy'], label='Training Accuracy', linewidth=2)
    axes[1].plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
    axes[1].set_xlabel('Epoch', fontsize=12)
    axes[1].set_ylabel('Accuracy', fontsize=12)
    axes[1].set_title('Training and Validation Accuracy (QAT)', fontsize=14)
    axes[1].legend(fontsize=11)
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Training history plot saved to {save_path}")

    plt.show()

## Train Quantized Model

In [24]:
# Train the model
history = train_model(
    model=model,
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    batch_size=32,
    max_epochs=50
)


Starting QAT training for up to 50 epochs...
Batch size: 32
Training samples: 40325
Validation samples: 20162

Note: EBOPs (Effective Bit-Operations) estimate FPGA resource usage.
Lower EBOPs = fewer hardware resources required.

Epoch 1/50


1261/1261 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.2456 - loss: 19.8154
Epoch 1: val_loss improved from None to 1.78078, saving model to best_quantized_model.keras

Epoch 1: finished saving model to best_quantized_model.keras
1261/1261 ━━━━━━━━━━━━━━━━━━━━ 27s 15ms/step - accuracy: 0.2575 - loss: 17.2461 - val_accuracy: 0.2460 - val_loss: 1.7808 - learning_rate: 0.0010 - ebops: 1260511.0000
Epoch 2/50
1256/1261 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.2531 - loss: 12.8293
Epoch 2: val_loss improved from 1.78078 to 1.74108, saving model to best_quantized_model.keras

Epoch 2: finished saving model to best_quantized_model.keras
1261/1261 ━━━━━━━━━━━━━━━━━━━━ 15s 12ms/step - accuracy: 0.2566 - loss: 11.3627 - val_accuracy: 0.2669 - val_loss: 1.7411 - learning_rate: 0.0010 - ebops: 775918.0000
Epoch 3/50
1259/1261 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.2618 - loss: 7.9389
Epoch 3: val_loss did not improve from 1.74108
1261/1261 ━━━━━━━━━━━━━━━━━━━━ 10s 8ms/step - accur

KeyboardInterrupt: 

In [ ]:
# Plot training history
plot_training_history(history, save_path='qat_training_history.png')

## Post-Training: Trace Min-Max

**Critical step before hls4ml conversion.**

When using `WRAP` overflow mode for activations (datalane), HGQ2 needs to know the min/max values of each layer's activations to determine the required integer bits. This is done by tracing the model on representative data.

**Important:** Use representative data (test set) for tracing, not training data.

In [ ]:
# Trace min/max values for WRAP overflow mode
# This determines the integer bits needed for each activation layer
print("Tracing min/max values for activation quantization...")
print("Using test set for representative statistics.\n")

# Use subset for tracing (faster)
trace_samples = min(5000, X_test.shape[0])
trace_minmax(model, X_test[:trace_samples], verbose=True)

print("\nMin/max tracing complete.")
print("Integer bits for each layer are now determined.")

## Evaluation

In [ ]:
# Load best model
print("Loading best quantized model...")
model.load_weights('best_quantized_model.keras')
print("Best model loaded successfully.\n")

# Evaluate on test set
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

In [ ]:
# Detailed classification report
y_pred_proba = model.predict(X_test)
y_pred = np.argmax(y_pred_proba, axis=1)
y_true = np.argmax(y_test, axis=1)

class_names = label_encoder.classes_

print("\nClassification Report (Quantized Model):\n")
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
ax.figure.colorbar(im, ax=ax)

# Add labels
ax.set(xticks=np.arange(cm.shape[1]),
       yticks=np.arange(cm.shape[0]),
       xticklabels=class_names,
       yticklabels=class_names,
       title='Confusion Matrix (Quantized Model)',
       ylabel='True label',
       xlabel='Predicted label')

# Rotate x-axis labels
plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

# Add text annotations
thresh = cm.max() / 2.
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, format(cm[i, j], 'd'),
                ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black")

plt.tight_layout()
plt.savefig('confusion_matrix_qat.png', dpi=300, bbox_inches='tight')
plt.show()

## Comparison with FP32 Base Model

Load the saved FP32 base model and compare performance.

In [ ]:
# Load FP32 base model
fp32_model_path = '../model/best_base_model.keras'

if os.path.exists(fp32_model_path):
    print("Loading FP32 base model for comparison...")
    fp32_model = keras.models.load_model(fp32_model_path)
    print("FP32 model loaded successfully.\n")
    
    # Evaluate FP32 model
    fp32_loss, fp32_accuracy = fp32_model.evaluate(X_test, y_test, verbose=0)
    fp32_pred_proba = fp32_model.predict(X_test)
    fp32_pred = np.argmax(fp32_pred_proba, axis=1)
    
    print("FP32 Base Model Performance:")
    print(f"  Test Loss: {fp32_loss:.4f}")
    print(f"  Test Accuracy: {fp32_accuracy:.4f}")
    
    print("\nQuantized Model Performance:")
    print(f"  Test Loss: {test_loss:.4f}")
    print(f"  Test Accuracy: {test_accuracy:.4f}")
    
    # Accuracy difference
    acc_diff = (test_accuracy - fp32_accuracy) * 100
    print(f"\nAccuracy difference: {acc_diff:+.2f}%")
else:
    print(f"FP32 model not found at {fp32_model_path}")
    print("Skipping comparison.")

In [ ]:
# Side-by-side comparison table
if os.path.exists(fp32_model_path):
    print("\n" + "="*60)
    print("PERFORMANCE COMPARISON: FP32 vs QAT")
    print("="*60)
    print(f"{'Metric':<30} {'FP32':>12} {'QAT':>12}")
    print("-"*60)
    print(f"{'Test Accuracy':<30} {fp32_accuracy:>12.4f} {test_accuracy:>12.4f}")
    print(f"{'Test Loss':<30} {fp32_loss:>12.4f} {test_loss:>12.4f}")
    
    # Model size
    fp32_params = fp32_model.count_params()
    qat_params = model.count_params()
    print(f"{'Parameters':<30} {fp32_params:>12,} {qat_params:>12,}")
    
    # Per-class accuracy comparison
    print("\n" + "="*60)
    print("PER-CLASS ACCURACY COMPARISON")
    print("="*60)
    print(f"{'Class':<15} {'FP32':>12} {'QAT':>12} {'Diff':>12}")
    print("-"*60)

    for i, class_name in enumerate(class_names):
        fp32_class_acc = np.mean(fp32_pred[y_true == i] == i)
        qat_class_acc = np.mean(y_pred[y_true == i] == i)
        diff = (qat_class_acc - fp32_class_acc) * 100
        print(f"{class_name:<15} {fp32_class_acc:>12.4f} {qat_class_acc:>12.4f} {diff:>+12.2f}%")

## hls4ml Conversion

Convert the quantized model to HLS (High-Level Synthesis) code for FPGA deployment.

In [ ]:
# hls4ml configuration
print("Configuring hls4ml conversion...\n")

# HLS configuration
# Note: HGQ2 models automatically invoke model-wise precision propagation
# Do NOT manually set precision config unless you know what you're doing
hls_config = {
    'Model': {
        'ReuseFactor': 1,  # Maximum parallelism
        'Strategy': 'Latency',  # Minimize latency
    }
}

# FPGA backend configuration
backend = 'Vitis'  # Recommended over Vivado for better Vivado HLS compatibility
io_type = 'io_parallel'  # Use io_stream for larger models

print(f"HLS Backend: {backend}")
print(f"I/O Type: {io_type}")
print(f"Reuse Factor: {hls_config['Model']['ReuseFactor']}")
print(f"Strategy: {hls_config['Model']['Strategy']}")
print("\nNote: Precision is automatically inferred from HGQ2 model.")

In [ ]:
# Convert model to HLS
print("\nConverting quantized model to HLS...\n")

output_dir = 'hls4ml_posture_cnn'

hls_model = hls4ml.converters.convert_from_keras_model(
    model,
    input_shapes=X_test[0:1].shape,
    hls_config=hls_config,
    output_dir=output_dir,
    backend=backend,
    io_type=io_type
)

print(f"\nHLS model created successfully.")
print(f"Output directory: {output_dir}")

In [ ]:
# Write HLS code to disk
print("Writing HLS C++ code to disk...")
hls_model.write()
print(f"HLS code written to {output_dir}/")
print(f"  - firmware/ : C++ source files")
print(f"  - project/ : Vivado/Vitis project files")

In [ ]:
# Compile HLS model for bit-exact simulation
print("\nCompiling HLS model for simulation...")
hls_model.compile()
print("HLS model compiled successfully.")

In [ ]:
# Verify bit-exactness between Keras and HLS models
print("\nVerifying bit-exactness between Keras and HLS models...\n")

# Use subset of test data for verification
n_samples = min(100, X_test.shape[0])

keras_predictions = model.predict(X_test[:n_samples])
hls_predictions = hls_model.predict(X_test[:n_samples])

# Check for mismatches
mismatches = np.sum(keras_predictions != hls_predictions)
total_values = np.prod(keras_predictions.shape)
match_percentage = (1 - mismatches / total_values) * 100

print(f"Verification Results:")
print(f"  Samples tested: {n_samples}")
print(f"  Mismatches: {mismatches} / {total_values} values")
print(f"  Match percentage: {match_percentage:.4f}%")

if mismatches == 0:
    print("\n✓ Perfect bit-exact match between Keras and HLS models!")
else:
    print(f"\n⚠ Minor mismatches detected ({mismatches} values).")
    print("  This may be due to floating-point precision differences.")
    print("  Verify that accuracy is maintained.")

In [ ]:
# Compare HLS predictions with true labels
hls_pred_classes = np.argmax(hls_predictions, axis=1)
hls_accuracy = np.mean(hls_pred_classes == y_true[:n_samples])

print(f"\nHLS Model Accuracy (on {n_samples} samples): {hls_accuracy:.4f}")
print(f"Keras Model Accuracy (on {n_samples} samples): {np.mean(np.argmax(keras_predictions, axis=1) == y_true[:n_samples]):.4f}")

## Summary

### Quantization-Aware Training Results

This notebook trained a quantized CNN using HGQ2's automatic bitwidth optimization. Key features:

1. **Automatic Quantization**: HGQ2 automatically optimizes bit-widths during training via gradient descent
2. **EBOPs Tracking**: Effective Bit-Operations provide resource estimates for FPGA deployment
3. **Bit-Exact HLS**: hls4ml generates RTL that matches the quantized model output

### Files Generated

- `best_quantized_model.keras`: Best trained QAT model
- `qat_training_history.png`: Training curves
- `confusion_matrix_qat.png`: Confusion matrix visualization
- `hls4ml_posture_cnn/`: HLS C++ code for FPGA synthesis

### Next Steps

1. Run Vivado/Vitis HLS synthesis on the generated C++ code
2. Estimate resource utilization (LUT, FF, DSP, BRAM)
3. Generate bitstream for PYNQ-Z2 target
4. Benchmark inference latency

In [ ]:
# Print final summary
print("\n" + "="*70)
print("QUANTIZED MODEL TRAINING SUMMARY")
print("="*70)
print(f"\n{'Architecture:':<25} 3 QConv2D + QBatchNorm + QDense")
print(f"{'Filter progression:':<25} [16, 16, 24]")
print(f"{'Total parameters:':<25} {model.count_params():,}")
print(f"\n{'Final Test Accuracy:':<25} {test_accuracy:.4f}")
print(f"{'Final Test Loss:':<25} {test_loss:.4f}")

if os.path.exists(fp32_model_path):
    print(f"\n{'FP32 Accuracy:':<25} {fp32_accuracy:.4f}")
    print(f"{'QAT Accuracy:':<25} {test_accuracy:.4f}")
    print(f"{'Accuracy Drop:':<25} {(fp32_accuracy - test_accuracy)*100:.2f}%")

print(f"\n{'HLS Output Directory:':<25} {output_dir}/")
print(f"{'Bit-exactness:':<25} {match_percentage:.4f}%")
print("="*70)